In [1]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [2]:
from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")


In [3]:
ROOT = Path.cwd()

CANDIDATES = [
    ROOT / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT.parent / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT / "notebooks" / "05_bivariate_eda_collections_macro_analysis.ipynb",
]

BIVARIATE_NOTEBOOK = next((p for p in CANDIDATES if p.exists()), None)

if BIVARIATE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Não encontrei 05_bivariate_eda_collections_macro_analysis.ipynb. "
        "Ajuste BIVARIATE_NOTEBOOK nesta célula."
    )

REPORT_DIR = ROOT / "reports"
CHART_DIR = REPORT_DIR / "collections_report_charts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = REPORT_DIR / "collections_macro_bivariate_analysis_report.pdf"

print("Notebook fonte :", BIVARIATE_NOTEBOOK.resolve())
print("PDF de saída   :", PDF_PATH.resolve())


Notebook fonte : C:\Users\beelt\Documents\collections_case_candidate\notebooks\05_bivariate_eda_collections_macro_analysis.ipynb
PDF de saída   : C:\Users\beelt\Documents\collections_case_candidate\notebooks\reports\collections_macro_bivariate_analysis_report.pdf


In [4]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


In [5]:
##_______________________________________________________________ "" ____________________________________________________________

In [6]:
# ============================================================
# DEEP ANALYSIS
# FIRST CONTACT × DPD × APP LOGIN × PAYMENT
#
# Question:
# Does recent app activity help identify the right timing
# for the FIRST collections message, beyond DPD?
#
# Population:
#   First observed WhatsApp per customer
#   DPD 1–30
#
# Outcomes:
#   Payment within 72h
#   Recovery / message
# ============================================================

import numpy as np
import pandas as pd

# ============================================================
# 1. BASE
# ============================================================

x = wa.copy()

x["sent_at"] = pd.to_datetime(x["sent_at"])

x["days_past_due"] = pd.to_numeric(
    x["days_past_due"],
    errors="coerce"
)

x["days_since_last_app_login"] = pd.to_numeric(
    x["days_since_last_app_login"],
    errors="coerce"
)

x["amount_paid_brl"] = pd.to_numeric(
    x["amount_paid_brl"],
    errors="coerce"
).fillna(0)

x["payment_72h"] = (
    x["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

x["recovery_72h"] = np.where(
    x["payment_72h"],
    x["amount_paid_brl"],
    0
)


# ============================================================
# 2. FIRST OBSERVED MESSAGE PER CUSTOMER
# ============================================================

first = (
    x.sort_values(
        ["customer_id", "sent_at", "message_id"]
    )
    .groupby("customer_id", as_index=False)
    .first()
)

# Early collections only
first = first.loc[
    first["days_past_due"].between(1, 30)
].copy()


print("=" * 100)
print("FIRST CONTACT — EARLY COLLECTIONS")
print("=" * 100)

print(f"Customers/messages : {len(first):,}")
print(
    f"Payment events     : "
    f"{first['payment_72h'].sum():,}"
)

print(
    f"Payment rate       : "
    f"{first['payment_72h'].mean():.2%}"
)

print(
    f"Recovery           : "
    f"R$ {first['recovery_72h'].sum():,.2f}"
)

print(
    f"Recovery / message : "
    f"R$ {first['recovery_72h'].mean():,.2f}"
)

print(
    f"Missing app login  : "
    f"{first['days_since_last_app_login'].isna().sum():,}"
)


# ============================================================
# 3. DISTRIBUTION OF APP LOGIN RECENCY
# ============================================================

print("\n" + "=" * 100)
print("APP LOGIN RECENCY — DISTRIBUTION")
print("=" * 100)

display(
    first["days_since_last_app_login"]
    .describe(
        percentiles=[
            .01, .05, .10, .25,
            .50, .75, .90, .95, .99
        ]
    )
    .to_frame("days_since_last_app_login")
)


# ============================================================
# 4. CREATE LOGIN BUCKETS
#
# Keep relatively granular initially.
# We can consolidate after seeing N.
# ============================================================

first["login_bucket"] = pd.cut(
    first["days_since_last_app_login"],
    bins=[
        -np.inf,
        0,
        1,
        3,
        7,
        14,
        30,
        60,
        np.inf
    ],
    labels=[
        "0d",
        "1d",
        "2–3d",
        "4–7d",
        "8–14d",
        "15–30d",
        "31–60d",
        "60+d"
    ]
)


# ============================================================
# 5. DPD BUCKET
# ============================================================

first["dpd_bucket"] = pd.cut(
    first["days_past_due"],
    bins=[0, 3, 7, 15, 30],
    labels=[
        "01–03",
        "04–07",
        "08–15",
        "16–30"
    ],
    include_lowest=True
)


# ============================================================
# 6. APP LOGIN → PAYMENT
# UNCONDITIONAL VIEW
# ============================================================

login_summary = (
    first
    .groupby("login_bucket", observed=True)
    .agg(
        messages=("customer_id", "size"),
        payments=("payment_72h", "sum"),
        payment_rate=("payment_72h", "mean"),
        total_recovery=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        avg_dpd=("days_past_due", "mean"),
        median_dpd=("days_past_due", "median"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("APP LOGIN RECENCY → PAYMENT")
print("FIRST CONTACT | DPD 1–30")
print("=" * 100)

display(
    login_summary.style.format({
        "payment_rate": "{:.2%}",
        "total_recovery": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_dpd": "{:.1f}",
        "median_dpd": "{:.1f}",
        "avg_balance": "R$ {:,.2f}"
    })
)


# ============================================================
# 7. DPD → PAYMENT
# ============================================================

dpd_summary = (
    first
    .groupby("dpd_bucket", observed=True)
    .agg(
        messages=("customer_id", "size"),
        payments=("payment_72h", "sum"),
        payment_rate=("payment_72h", "mean"),
        total_recovery=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        avg_login_recency=("days_since_last_app_login", "mean"),
        median_login_recency=("days_since_last_app_login", "median")
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("DPD → PAYMENT")
print("FIRST CONTACT")
print("=" * 100)

display(
    dpd_summary.style.format({
        "payment_rate": "{:.2%}",
        "total_recovery": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_login_recency": "{:.1f}",
        "median_login_recency": "{:.1f}"
    })
)


# ============================================================
# 8. CRITICAL ANALYSIS:
# DPD × APP LOGIN
# ============================================================

cross = (
    first
    .groupby(
        ["dpd_bucket", "login_bucket"],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        payments=("payment_72h", "sum"),
        payment_rate=("payment_72h", "mean"),
        recovery=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean")
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("DPD × APP LOGIN → PAYMENT")
print("=" * 100)

display(
    cross.style.format({
        "payment_rate": "{:.2%}",
        "recovery": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}"
    })
)


# ============================================================
# 9. HEATMAP TABLE — PAYMENT RATE
# ============================================================

payment_matrix = (
    cross.pivot(
        index="dpd_bucket",
        columns="login_bucket",
        values="payment_rate"
    )
)

print("\n" + "=" * 100)
print("PAYMENT RATE 72H")
print("=" * 100)

display(
    payment_matrix.style
    .format("{:.1%}")
    .background_gradient(axis=None)
)


# ============================================================
# 10. HEATMAP TABLE — RECOVERY / MESSAGE
# ============================================================

recovery_matrix = (
    cross.pivot(
        index="dpd_bucket",
        columns="login_bucket",
        values="recovery_per_message"
    )
)

print("\n" + "=" * 100)
print("RECOVERY / MESSAGE")
print("=" * 100)

display(
    recovery_matrix.style
    .format("R$ {:.2f}")
    .background_gradient(axis=None)
)


# ============================================================
# 11. HEATMAP TABLE — SAMPLE SIZE
# Essential to avoid interpreting tiny cells
# ============================================================

n_matrix = (
    cross.pivot(
        index="dpd_bucket",
        columns="login_bucket",
        values="messages"
    )
)

print("\n" + "=" * 100)
print("SAMPLE SIZE — N")
print("=" * 100)

display(
    n_matrix.style.format("{:,.0f}")
)


# ============================================================
# 12. DAILY DPD
#
# Don't impose buckets yet:
# inspect DPD 1, 2, 3...30
# ============================================================

daily = (
    first
    .groupby("days_past_due")
    .agg(
        messages=("customer_id", "size"),
        payments=("payment_72h", "sum"),
        payment_rate=("payment_72h", "mean"),
        recovery_per_message=("recovery_72h", "mean"),
        median_login_recency=("days_since_last_app_login", "median")
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("DAILY DPD — FIRST CONTACT")
print("=" * 100)

display(
    daily.style.format({
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "median_login_recency": "{:.1f}"
    })
)

FIRST CONTACT — EARLY COLLECTIONS
Customers/messages : 11,723
Payment events     : 1,155
Payment rate       : 9.85%
Recovery           : R$ 762,592.41
Recovery / message : R$ 65.05
Missing app login  : 0

APP LOGIN RECENCY — DISTRIBUTION


,days_since_last_app_login
count,"11,723.00"
mean,19.03
std,20.02
min,0.00
1%,0.00
5%,1.00
10%,2.00
25%,5.00
50%,13.00
75%,26.00



APP LOGIN RECENCY → PAYMENT
FIRST CONTACT | DPD 1–30


,login_bucket,messages,payments,payment_rate,total_recovery,recovery_per_message,avg_dpd,median_dpd,avg_balance
0,0d,327,37,11.31%,"R$ 22,041.39",R$ 67.40,3.2,2.0,R$ 853.45
1,1d,647,63,9.74%,"R$ 35,474.16",R$ 54.83,3.1,2.0,R$ 844.05
2,2–3d,1092,115,10.53%,"R$ 66,508.85",R$ 60.91,3.0,2.0,R$ 832.52
3,4–7d,1891,201,10.63%,"R$ 139,730.35",R$ 73.89,3.0,2.0,R$ 833.39
4,8–14d,2419,255,10.54%,"R$ 163,797.90",R$ 67.71,3.0,2.0,R$ 856.57
5,15–30d,3020,271,8.97%,"R$ 188,478.05",R$ 62.41,3.1,2.0,R$ 861.89
6,31–60d,1793,179,9.98%,"R$ 124,952.25",R$ 69.69,3.0,2.0,R$ 854.08
7,60+d,534,34,6.37%,"R$ 21,609.46",R$ 40.47,3.2,3.0,R$ 839.24



DPD → PAYMENT
FIRST CONTACT


...

login_bucket,0d,1d,2–3d,4–7d,8–14d,15–30d,31–60d,60+d
dpd_bucket,,,,,,,,
01–03,R$ 66.06,R$ 56.94,R$ 56.76,R$ 78.56,R$ 68.31,R$ 63.60,R$ 70.72,R$ 39.51
04–07,R$ 47.09,R$ 41.38,R$ 68.84,R$ 71.21,R$ 66.63,R$ 60.23,R$ 57.21,R$ 39.17
08–15,R$ 142.83,R$ 95.41,R$ 66.61,R$ 28.39,R$ 66.21,R$ 58.37,R$ 118.34,R$ 57.16
16–30,R$ 180.88,R$ 0.00,R$ 281.80,R$ 62.75,R$ 0.00,R$ 68.90,R$ 0.00,R$ 0.00



SAMPLE SIZE — N


login_bucket,0d,1d,2–3d,4–7d,8–14d,15–30d,31–60d,60+d
dpd_bucket,,,,,,,,
01–03,221,430,759,"1,301","1,708","2,044","1,253",352
04–07,82,178,270,482,577,770,445,147
08–15,20,38,60,104,131,198,92,34
16–30,4,1,3,4,3,8,3,1



DAILY DPD — FIRST CONTACT


,days_past_due,messages,payments,payment_rate,recovery_per_message,median_login_recency
0,1,3841,386,10.05%,R$ 66.88,12.0
1,2,2482,245,9.87%,R$ 67.79,13.0
2,3,1745,170,9.74%,R$ 62.14,13.0
3,4,1241,128,10.31%,R$ 71.77,13.0
4,5,822,73,8.88%,R$ 57.10,13.0
5,6,558,48,8.60%,R$ 48.58,12.0
6,7,330,31,9.39%,R$ 51.71,13.0
7,8,228,23,10.09%,R$ 67.34,13.0
8,9,162,15,9.26%,R$ 63.26,15.0
9,10,101,12,11.88%,R$ 65.53,11.0


In [7]:
# ============================================================
# SHARE OF TOTAL RECOVERY BY DPD
# FIRST CONTACT | DPD 1–30
# ============================================================

import numpy as np
import pandas as pd

# first já foi construído anteriormente:
# 1 linha = primeiro contato observado por cliente
# DPD 1–30
# recovery_72h = amount_paid_brl quando paid_within_72h == True

recovery_by_dpd = (
    first
    .groupby("days_past_due", as_index=False)
    .agg(
        customers=("customer_id", "size"),
        payers=("payment_72h", "sum"),
        recovered_brl=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        payment_rate=("payment_72h", "mean")
    )
)

# ------------------------------------------------------------
# Total recovery
# ------------------------------------------------------------

total_recovery = recovery_by_dpd["recovered_brl"].sum()

# ------------------------------------------------------------
# Share of ALL recovered money
# ------------------------------------------------------------

recovery_by_dpd["recovery_share"] = (
    recovery_by_dpd["recovered_brl"] / total_recovery
)

# cumulative share
recovery_by_dpd["cumulative_recovery_share"] = (
    recovery_by_dpd["recovery_share"].cumsum()
)

# customer volume share — useful comparison
total_customers = recovery_by_dpd["customers"].sum()

recovery_by_dpd["customer_share"] = (
    recovery_by_dpd["customers"] / total_customers
)

# Difference:
# Is this DPD concentrating more/less recovery than volume?
recovery_by_dpd["recovery_vs_volume_pp"] = (
    recovery_by_dpd["recovery_share"]
    - recovery_by_dpd["customer_share"]
)

print("=" * 110)
print("SHARE OF TOTAL RECOVERY BY DPD — FIRST CONTACT")
print("=" * 110)

print(f"Total customers : {total_customers:,}")
print(f"Total recovery  : R$ {total_recovery:,.2f}")

display(
    recovery_by_dpd.style.format({
        "customers": "{:,.0f}",
        "payers": "{:,.0f}",
        "recovered_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "payment_rate": "{:.2%}",
        "recovery_share": "{:.2%}",
        "cumulative_recovery_share": "{:.2%}",
        "customer_share": "{:.2%}",
        "recovery_vs_volume_pp": "{:+.2%}"
    })
)

SHARE OF TOTAL RECOVERY BY DPD — FIRST CONTACT
Total customers : 11,723
Total recovery  : R$ 762,592.41


,days_past_due,customers,payers,recovered_brl,recovery_per_message,payment_rate,recovery_share,cumulative_recovery_share,customer_share,recovery_vs_volume_pp
0,1,"3,841",386,"R$ 256,870.70",R$ 66.88,10.05%,33.68%,33.68%,32.76%,+0.92%
1,2,"2,482",245,"R$ 168,244.26",R$ 67.79,9.87%,22.06%,55.75%,21.17%,+0.89%
2,3,"1,745",170,"R$ 108,441.01",R$ 62.14,9.74%,14.22%,69.97%,14.89%,-0.67%
3,4,"1,241",128,"R$ 89,063.03",R$ 71.77,10.31%,11.68%,81.65%,10.59%,+1.09%
4,5,822,73,"R$ 46,936.31",R$ 57.10,8.88%,6.15%,87.80%,7.01%,-0.86%
5,6,558,48,"R$ 27,109.58",R$ 48.58,8.60%,3.55%,91.35%,4.76%,-1.20%
6,7,330,31,"R$ 17,063.32",R$ 51.71,9.39%,2.24%,93.59%,2.81%,-0.58%
7,8,228,23,"R$ 15,352.95",R$ 67.34,10.09%,2.01%,95.61%,1.94%,+0.07%
8,9,162,15,"R$ 10,247.51",R$ 63.26,9.26%,1.34%,96.95%,1.38%,-0.04%
9,10,101,12,"R$ 6,618.68",R$ 65.53,11.88%,0.87%,97.82%,0.86%,+0.01%


In [8]:
# ============================================================
# SHARE OF TOTAL RECOVERY BY DPD
# ALL HISTORICAL RECOVERY
# ============================================================

import numpy as np
import pandas as pd

x = wa.copy()

x["days_past_due"] = pd.to_numeric(
    x["days_past_due"],
    errors="coerce"
)

x["amount_paid_brl"] = pd.to_numeric(
    x["amount_paid_brl"],
    errors="coerce"
).fillna(0)

x["payment_72h"] = (
    x["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

# Recovery attributed to the message
x["recovery_72h"] = np.where(
    x["payment_72h"],
    x["amount_paid_brl"],
    0
)

# ============================================================
# DPD 1–30
# ============================================================

early = x.loc[
    x["days_past_due"].between(1, 30)
].copy()

# ============================================================
# RECOVERY BY DPD
# ============================================================

recovery_dpd = (
    early
    .groupby("days_past_due", as_index=False)
    .agg(
        messages=("message_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_72h", "sum"),
        recovered_brl=("recovery_72h", "sum")
    )
)

# ============================================================
# IMPORTANT:
# denominator = ALL recovery in the entire WA history
# ============================================================

total_recovery_all = x["recovery_72h"].sum()

recovery_dpd["share_total_recovery"] = (
    recovery_dpd["recovered_brl"]
    / total_recovery_all
)

recovery_dpd["cumulative_share_total"] = (
    recovery_dpd["share_total_recovery"].cumsum()
)

# Also useful:
# share only within DPD 1–30
total_recovery_early = early["recovery_72h"].sum()

recovery_dpd["share_early_recovery"] = (
    recovery_dpd["recovered_brl"]
    / total_recovery_early
)

recovery_dpd["cumulative_share_early"] = (
    recovery_dpd["share_early_recovery"].cumsum()
)

print("=" * 110)
print("RECOVERY SHARE BY DPD — ALL HISTORICAL PAYMENTS")
print("=" * 110)

print(f"TOTAL WA RECOVERY      : R$ {total_recovery_all:,.2f}")
print(f"RECOVERY DPD 1–30      : R$ {total_recovery_early:,.2f}")
print(
    f"% recovered by DPD 30 : "
    f"{total_recovery_early / total_recovery_all:.2%}"
)

display(
    recovery_dpd.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "recovered_brl": "R$ {:,.2f}",
        "share_total_recovery": "{:.2%}",
        "cumulative_share_total": "{:.2%}",
        "share_early_recovery": "{:.2%}",
        "cumulative_share_early": "{:.2%}"
    })
)

RECOVERY SHARE BY DPD — ALL HISTORICAL PAYMENTS
TOTAL WA RECOVERY      : R$ 3,459,305.30
RECOVERY DPD 1–30      : R$ 3,028,175.92
% recovered by DPD 30 : 87.54%


,days_past_due,messages,customers,payment_events,recovered_brl,share_total_recovery,cumulative_share_total,share_early_recovery,cumulative_share_early
0,1,"3,841","3,841",386,"R$ 256,870.70",7.43%,7.43%,8.48%,8.48%
1,2,"3,632","3,632",346,"R$ 233,301.69",6.74%,14.17%,7.70%,16.19%
2,3,"3,427","3,427",330,"R$ 205,822.52",5.95%,20.12%,6.80%,22.98%
3,4,"3,341","3,341",298,"R$ 205,215.71",5.93%,26.05%,6.78%,29.76%
4,5,"3,276","3,276",275,"R$ 183,204.59",5.30%,31.35%,6.05%,35.81%
5,6,"3,115","3,115",279,"R$ 167,880.37",4.85%,36.20%,5.54%,41.35%
6,7,"3,092","3,092",236,"R$ 148,801.59",4.30%,40.50%,4.91%,46.27%
7,8,"3,063","3,063",265,"R$ 169,760.96",4.91%,45.41%,5.61%,51.87%
8,9,"2,889","2,889",247,"R$ 147,154.55",4.25%,49.66%,4.86%,56.73%
9,10,"2,809","2,809",218,"R$ 130,811.38",3.78%,53.44%,4.32%,61.05%


In [ ]:
# ============================================================
# BEST DAY FOR FIRST CONTACT — DPD 1–7
# Pairwise permutation tests + bootstrap CI + Holm correction
# ============================================================

import numpy as np
import pandas as pd
from itertools import combinations

# ------------------------------------------------------------
# Population
# first = one row per customer, first observed WA message
# ------------------------------------------------------------

d = first.loc[
    first["days_past_due"].between(1, 7)
].copy()

d["days_past_due"] = d["days_past_due"].astype(int)

# ------------------------------------------------------------
# Functions
# ------------------------------------------------------------

def permutation_test_mean(a, b, n_perm=50_000, seed=42):

    rng = np.random.default_rng(seed)

    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    observed = a.mean() - b.mean()

    pooled = np.concatenate([a, b])
    n_a = len(a)

    count = 0

    for _ in range(n_perm):

        perm = rng.permutation(pooled)

        diff = (
            perm[:n_a].mean()
            - perm[n_a:].mean()
        )

        if abs(diff) >= abs(observed):
            count += 1

    p = (count + 1) / (n_perm + 1)

    return observed, p


def bootstrap_diff(a, b, n_boot=20_000, seed=42):

    rng = np.random.default_rng(seed)

    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    diffs = np.empty(n_boot)

    for i in range(n_boot):

        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)

        diffs[i] = aa.mean() - bb.mean()

    return np.percentile(
        diffs,
        [2.5, 97.5]
    )


# ------------------------------------------------------------
# Pairwise comparisons
# ------------------------------------------------------------

results = []

for day_a, day_b in combinations(range(1, 8), 2):

    a = d.loc[
        d["days_past_due"].eq(day_a),
        "recovery_72h"
    ].values

    b = d.loc[
        d["days_past_due"].eq(day_b),
        "recovery_72h"
    ].values

    diff, p = permutation_test_mean(a, b)

    ci_low, ci_high = bootstrap_diff(a, b)

    results.append({
        "day_A": day_a,
        "day_B": day_b,

        "N_A": len(a),
        "N_B": len(b),

        "RPM_A": a.mean(),
        "RPM_B": b.mean(),

        "diff_A_minus_B": diff,

        "uplift_A_vs_B":
            diff / b.mean()
            if b.mean() != 0 else np.nan,

        "CI_low": ci_low,
        "CI_high": ci_high,

        "p_raw": p
    })


res = pd.DataFrame(results)


# ============================================================
# HOLM CORRECTION
# 21 pairwise comparisons
# ============================================================

m = len(res)

order = np.argsort(res["p_raw"].values)

adjusted = np.empty(m)

running_max = 0

for rank, idx in enumerate(order):

    adj = min(
        (m - rank) * res.loc[idx, "p_raw"],
        1
    )

    running_max = max(running_max, adj)

    adjusted[idx] = running_max

res["p_holm"] = adjusted

res["significant_5pct"] = (
    res["p_holm"] < 0.05
)


# ============================================================
# DISPLAY
# ============================================================

res = res.sort_values(
    ["p_holm", "p_raw"]
).reset_index(drop=True)

display(
    res.style.format({
        "RPM_A": "R$ {:,.2f}",
        "RPM_B": "R$ {:,.2f}",
        "diff_A_minus_B": "R$ {:+,.2f}",
        "uplift_A_vs_B": "{:+.1%}",
        "CI_low": "R$ {:+,.2f}",
        "CI_high": "R$ {:+,.2f}",
        "p_raw": "{:.4f}",
        "p_holm": "{:.4f}"
    })
)